<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 2</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/5185082/pexels-photo-5185082.jpeg" width=720></a>
<center><font size=6>LLM-Powered Research Assistant</center></font>

# Problem Statement

## Business Context

In today's competitive and rapidly evolving technological landscape, research and development teams are often overwhelmed by the volume of new scientific publications. Manually reviewing, summarizing, and extracting key insights from these documents is time-consuming and can delay critical innovation decisions. To address this challenge, a technology consulting firm is building an LLM-powered assistant designed to streamline the literature review process. This AI assistant will automatically extract summaries, metadata, and emerging trends from collections of research papers, significantly reducing manual effort and accelerating the discovery of relevant information. By applying this solution to a curated dataset of recent publications, the firm aims to demonstrate how advanced language models can enhance productivity and support timely, data-driven innovation across research-intensive domains.

##  Objective

The goal is to develop a prototype that demonstrates how Natural Language Processing (NLP) can support research teams in efficiently querying large scientific documents using Retrieval-Augmented Generation (RAG).

Specifically, the system aims to:

* Answer user questions by retrieving relevant content from long research papers.
* Support natural-language queries without requiring users to read the entire document.
* Simulate an intelligent assistant that simplifies literature review workflows.

This case study focuses on building a small part of that broader solution, a system where users upload lengthy research PDFs, and the model provides grounded, relevant answers using RAG. This enables researchers to extract targeted information faster and make informed decisions more quickly.

Through successful implementation, the organization seeks to improve research productivity, reduce time spent on manual document review, and accelerate innovation.


## Data Description

The dataset consists of four research papers in PDF format focused on Prompt Engineering.

Here are the research paper names and links:

1. **Language Models are Few-Shot Learners**
   *Authors: Tom B. Brown et al. (OpenAI)*

2. **The Power of Scale for Parameter-Efficient Prompt Tuning**
   *Authors: Brian Lester, Rami Al-Rfou, Noah Constant (Google Research)*

3. **AutoPrompt: Eliciting Knowledge from Language Models with Automatically Generated Prompts**
   *Authors: Taylor Shin, Yasaman Razeghi, Robert L. Logan IV, Eric Wallace, Sameer Singh*

4. **Prompt Programming for Large Language Models: Beyond the Few-Shot Paradigm**
   *Authors: Andy J. W. Reynolds, Sam McDonell (Stanford)*



# Installing and Importing Necessary Libraries

In [ ]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 11.7 MB/s eta 0:00

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

# Question Answering using LLM

### OpenAI API Calling



In [ ]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

### Defining the function to Generate a Response From the LLM

In [ ]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.3, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [ ]:
# Define the question
question_1 = "What is the difference between zero-shot and few-shot prompting in language models?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

'Zero-shot and few-shot prompting are two techniques used to interact with language models, particularly in the context of natural language processing tasks. Here’s a breakdown of the differences between the two:\n\n### Zero-Shot Prompting\n- **Definition**: In zero-shot prompting, the model is asked to perform a task without any specific examples or prior demonstrations. The prompt typically includes a clear instruction or question, but no examples of input-output pairs.\n- **Example**: If you want the model to summarize a text, you might simply ask, "Summarize the following text: [text]." There are no examples provided to guide the model on how to structure the summary.\n- **Use Case**: This approach is useful when you want to test the model\'s ability to generalize from its training data without any specific guidance or when examples are not available.\n\n### Few-Shot Prompting\n- **Definition**: In few-shot prompting, the model is provided with a small number of examples (usually o

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [ ]:
question_2 = "How does the performance of in-context learning change based on the number of examples provided and the nature of the task?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

'In-context learning (ICL) refers to the ability of models, particularly large language models, to adapt to new tasks by conditioning on examples provided in the input context. The performance of ICL can vary significantly based on several factors, including the number of examples provided and the nature of the task. Here’s how these factors can influence performance:\n\n### Number of Examples\n\n1. **Few-Shot Learning**: \n   - With a small number of examples (e.g., 1-5), models can often generalize well, especially if the examples are representative of the task. However, the performance may be inconsistent due to the limited context.\n   - As the number of examples increases, models typically show improved performance, as they have more information to draw from, which helps in understanding the task better.\n\n2. **Scaling Effects**: \n   - There is often a diminishing return on performance as more examples are added. After a certain point, additional examples may not significantly e

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [ ]:
question_3 = "What are the limitations of discrete prompts, and how to overcome them?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"Discrete prompts, which are specific and clearly defined inputs given to a model like me, can have several limitations. Here are some of the key limitations along with strategies to overcome them:\n\n### Limitations of Discrete Prompts\n\n1. **Lack of Context**:\n   - Discrete prompts may not provide enough context for the model to generate a relevant or nuanced response. This can lead to misunderstandings or overly simplistic answers.\n\n2. **Ambiguity**:\n   - If a prompt is vague or ambiguous, the model may interpret it in multiple ways, leading to responses that do not align with the user's intent.\n\n3. **Inflexibility**:\n   - Discrete prompts can limit the model's ability to explore related topics or generate creative responses, as they often constrain the conversation to a narrow scope.\n\n4. **Overfitting to Prompt Structure**:\n   - Models may become overly reliant on the structure of the prompt, leading to repetitive or formulaic responses that lack originality.\n\n5. **Dif

**Observations**

**Question 1: Zero-Shot vs Few-Shot Prompting**

The Base Prompt answer explains the concepts clearly with examples and use cases, making the distinction between zero-shot and few-shot easy to understand; however, it is slightly repetitive and does not mention performance differences or limitations of each approach.

**Question 2: In-Context Learning Performance**

The answer effectively highlights how the number of examples and task nature affect performance, showing awareness of task complexity and example relevance.

**Question 3: Limitations of Discrete Prompts**

The Base Prompt lists key limitations and strategies to overcome them, giving practical guidance for prompt design.

## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [ ]:
system_prompt = """
You are an AI assistant developed for a technology consulting firm to support Research and Development (R&D) teams in efficiently analyzing large volumes of scientific publications.

Your primary goal is to streamline the literature review process by:
- Extracting accurate and concise summaries from research papers.
- Identifying key metadata such as authors, publication year, research domain, and methodology.
- Highlighting emerging trends, insights, and innovations across related works.

When responding:
- Maintain factual accuracy and clarity at all times.
- Present insights in a structured, business-relevant, and easy-to-understand format.
- Avoid speculation or assumptions beyond the provided research content.
- If a query requires information not available in the given papers, acknowledge the limitation instead of inferring.

The objective is to help researchers and decision-makers quickly discover relevant information, reduce manual review time, and accelerate data-driven innovation.
"""


### Defining the function to Generate a Response From the LLM

In [ ]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [ ]:
# Define the question
question_1 = "What is the difference between zero-shot and few-shot prompting in language models?"
base_prompt_response_1 = response(system_prompt,question_1)
base_prompt_response_1

'Zero-shot and few-shot prompting are two techniques used to interact with language models, particularly in the context of natural language processing tasks. Here’s a breakdown of the differences between the two:\n\n### Zero-Shot Prompting\n- **Definition**: In zero-shot prompting, the model is asked to perform a task without any specific examples or prior demonstrations. The prompt typically includes a clear instruction or question, but no examples of input-output pairs.\n- **Example**: If you want the model to summarize a text, you might simply ask, "Summarize the following text: [text]." There are no examples provided to guide the model on how to structure the summary.\n- **Use Case**: This approach is useful when you want to test the model\'s ability to generalize from its training data without any specific guidance or when examples are not available.\n\n### Few-Shot Prompting\n- **Definition**: In few-shot prompting, the model is provided with a small number of examples (usually o

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [ ]:
question_2 = "How does the performance of in-context learning change based on the number of examples provided and the nature of the task?"
base_prompt_response_2 = response(system_prompt,question_2)
base_prompt_response_2

'In-context learning (ICL) refers to the ability of models, particularly large language models, to adapt to new tasks by conditioning on examples provided in the input context. The performance of ICL can vary significantly based on several factors, including the number of examples provided and the nature of the task. Here’s how these factors can influence performance:\n\n### Number of Examples\n\n1. **Few-Shot Learning**: \n   - With a small number of examples (e.g., 1-5), models can often generalize well, especially if the examples are representative of the task. However, the performance may be inconsistent due to the limited context.\n   - As the number of examples increases, models typically show improved performance, as they have more information to draw from, which helps in understanding the task better.\n\n2. **Scaling Effects**: \n   - There is often a diminishing return on performance as more examples are added. After a certain point, additional examples may not significantly e

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [ ]:
question_3 = "What are the limitations of discrete prompts, and how to overcome them?"
base_prompt_response_3 = response(system_prompt,question_3)
base_prompt_response_3

"Discrete prompts, which are specific and clearly defined inputs given to a model like me, can have several limitations. Here are some of the key limitations along with strategies to overcome them:\n\n### Limitations of Discrete Prompts\n\n1. **Lack of Context**:\n   - Discrete prompts may not provide enough context for the model to generate a relevant or nuanced response. This can lead to misunderstandings or overly simplistic answers.\n\n2. **Ambiguity**:\n   - If a prompt is vague or ambiguous, the model may interpret it in multiple ways, leading to responses that do not align with the user's intent.\n\n3. **Inflexibility**:\n   - Discrete prompts can limit the model's ability to explore related topics or generate creative responses, as they often constrain the conversation to a narrow scope.\n\n4. **Overfitting to Prompt Structure**:\n   - Models may become overly reliant on the structure of the prompt, leading to repetitive or formulaic responses that lack originality.\n\n5. **Dif

**Observation**

**Question 1 (Zero-Shot vs Few-Shot Prompting)**

The answer is well-structured with clear definitions, examples, and use cases, showing good instructional prompting.

**Question 2 (In-Context Learning Performance)**

The response demonstrates good depth and covers multiple influencing factors, but it becomes verbose, includes repeated ideas.

**Question 3 (Limitations of Discrete Prompts)**

The answer is comprehensive and lists both limitations and solutions clearly, but it is too lengthy, has overlapping points.

# Data Preparation for RAG

## Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
! unzip "/content/ResearchPaper.zip"

Archive:  /content/ResearchPaper.zip
   creating: ResearchPaper/
  inflating: ResearchPaper/AutoPrompt Eliciting Knowledge from Language Models with Automatically Generated Prompts.pdf  
  inflating: ResearchPaper/Language Models are Few-Shot Learners.pdf  
  inflating: ResearchPaper/Prompt Programming for Large Language Models.pdf  
  inflating: ResearchPaper/The Power of Scale for Parameter-Efficient Prompt Tuning.pdf  


In [ ]:
# Uploading multiple pdfs:
from glob import glob

# Path to folder with PDFs
DOC_FOLDER = "ResearchPaper/"
pdf_files = glob(DOC_FOLDER + "*.pdf")

In [ ]:
pdf_files

['ResearchPaper/AutoPrompt Eliciting Knowledge from Language Models with Automatically Generated Prompts.pdf',
 'ResearchPaper/Language Models are Few-Shot Learners.pdf',
 'ResearchPaper/Prompt Programming for Large Language Models.pdf',
 'ResearchPaper/The Power of Scale for Parameter-Efficient Prompt Tuning.pdf']

In [ ]:
# Load each PDF file and store the documents
all_documents = []
for pdf_file in pdf_files:
    loader = PyMuPDFLoader(pdf_file)
    documents = loader.load()
    all_documents.extend(documents)


## Data Chunking

### Chunk the PDF into Manageable Text Sections

In [ ]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=512,                                                             # Each chunk will have up to 512 character
)

### Split the Loaded PDF into Chunks for Further Processing

In [ ]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = text_splitter.split_documents(all_documents)                  # Returns a list of chunked documents

### Check the Number of Chunks Created

In [ ]:
print(f"Created {len(document_chunks)} chunks.")

Created 355 chunks.


## Generate Vector Embeddings for Text Chunks Using OpenAI

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


Below is a sample code snippet you can refer to for using an open-source embedding model.

In [ ]:
# file_name = 'config.json'                                                       # Name of the configuration file
# with open(file_name, 'r') as file:                                              # Open the config file in read mode
#     config = json.load(file)                                                    # Load the JSON content as a dictionary
#     HF_TOKEN = config.get("HF_TOKEN")


# # Store API credentials in environment variables
# os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
# from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# # Use HuggingFaceEmbeddings wrapper with SentenceTransformer model
# embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')


## Vector Database Creation

In [ ]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model                                                             # Embedding model for converting text to vectors
 )

## Retrieval and Response Generation using Vector Search

### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [ ]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 5}                                                      # Retrieve top 5 most relevant documents
)

## System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support research teams in efficiently reviewing scientific literature. Your task is to provide evidence-based, concise, and relevant summaries based on the context provided from research papers.

User input will include the necessary context for you to answer their questions. This context will begin with the token:

###Context
The context contains excerpts from one or more research papers, along with associated metadata such as titles, authors, abstracts, keywords, and specific sections relevant to the query.

When crafting your response
-Use only the provided context to answer the question.
-If the answer is found in the context, respond with concise and insight-focused summaries.
-Include the paper title and, where applicable, arXiv ID or section reference as the source.
-If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


Please adhere to the following response guidelines:
-Provide clear, direct answers using only the given context.
-Do not include any additional information outside of the context.
-Avoid rephrasing or generalizing unless explicitly relevant to the question.
-If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
-If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."


Here is an example of how to structure your response:

Answer:
[Answer based on context]

Source:
[Source details with page or section]
"""

In [ ]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from GEN AI Research Paper and their sources that are relevant to the Gen AI question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=5,max_tokens=500,temperature=0.3,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

#### Below are sample code snippets you can refer to for downloading the open source model and implementing the function to generate a RAG-based response.

In [ ]:
# !pip install transformers torch

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# # Define model name from Hugging Face
# MISTRAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_NAME,token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL_NAME,token=HF_TOKEN)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

In [ ]:
# def generate_rag_response_mistral(user_input, k=5, max_tokens=500, temperature=0.7, top_p=0.95):
#     global qna_system_message, qna_user_message_template, retriever, model, tokenizer

#     try:

#         relevant_chunks = retriever.get_relevant_documents(query=user_input, k=k)
#         context_list = [doc.page_content for doc in relevant_chunks]
#         context = ". ".join(context_list)


#         user_message = qna_user_message_template.replace("{context}", context)
#         user_message = user_message.replace("{question}", user_input)
#         prompt = f"{qna_system_message.strip()}\n\n{user_message.strip()}"


#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_tokens,
#             temperature=temperature,
#             top_p=top_p,
#             do_sample=True
#         )

#         response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

#     except Exception as e:
#         response = f"Sorry, I encountered the following error:\n{e}"

#     return response


# Question Answering using RAG

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipython-input-365345968.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


"Answer:\nZero-shot prompting involves providing the model only with a natural language instruction describing the task, without any demonstrations. This method is challenging and can be ambiguous, as it relies solely on the model's understanding from pre-training. In contrast, few-shot prompting includes a few examples along with the task description, allowing the model to perform better by leveraging these demonstrations. Few-shot prompting typically leads to improved performance compared to zero-shot, as it provides context that can guide the model's responses.\n\nSource:\nGEN AI Research Paper, Sections 2.1-2.4"

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

"The performance of in-context learning improves significantly with the number of examples provided. As demonstrated in the context, larger models exhibit steeper learning curves, indicating that they are more proficient at in-context learning when given more examples. Specifically, the few-shot learning performance improves dramatically with the addition of examples in the model's context. For instance, in the evaluation of GPT-3, performance on tasks increases with the number of demonstrations, showing that both the model size and the number of examples positively influence learning outcomes.\n\nAdditionally, the nature of the task also plays a role; while GPT-3 achieves promising results in zero-shot and one-shot settings, its performance in the few-shot setting can sometimes be competitive with or even surpass state-of-the-art results. However, the effectiveness of in-context learning still lags behind traditional fine-tuning methods.\n\nSource:\nGEN AI Research Paper, Figures 1.2,

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Answer:\nThe limitations of discrete prompts include the difficulty of designing prompts for specific tasks and the lack of automated methods to create effective prompts. Task-agnostic prompts are often less effective than those tailored to a specific task, requiring significant human time investment. To overcome these limitations, the paper suggests creating automated methods to generate task-specific prompts, such as using metaprompts. Metaprompts encapsulate a general intention that can unfold into specific prompts when combined with additional information, thereby enhancing the effectiveness of the prompts.\n\nSource:\nGEN AI Research Paper, §4.7 Metaprompt programming'

**Observation**

The answers generated from RAG are concise, well-structured, and include source references, which adds credibility and makes them trustworthy. Overall, they effectively capture key concepts in prompt engineering, providing clear explanations and relevant context, making them high-quality responses.

# Conclusion

<font size=4>Business Impact</font>

* **Enhanced Decision-Making**: Quick access to key insights and emerging trends empowers research teams to make informed, timely decisions.
* **Scalable Knowledge Management**: Structured outputs enable better indexing, retrieval, and organization of research content, improving long-term knowledge curation.

This approach lays the groundwork for building intelligent research support systems that are scalable, context-aware, and aligned with the dynamic needs of innovation-driven industries.

<font size=4>Improvement Areas</font>

* The system currently depends on the quality and consistency of input PDFs and may struggle with poorly formatted or scanned documents.
* While the LLM captures high-level insights, deeper domain-specific interpretations may require fine-tuning or human validation.

<font size=6 color='#4682B4'>Power Ahead</font>
___

# [OPTIONAL] Deployment

The deployment section is only meant to demonstrate the research assistant in action.


## Streamlit on Hugging Face

### app.py

**NOTE**: You can use this prompt to generate the code for buidling the Streamlit app. However, a few minor adjustments might be needed afterwards.

***Prompt***:

<font size=3 color="#4682B4"><b>Build a Streamlit app that lets users upload PDFs and ask questions, with answers generated via OpenAI using RAG.
</font>

In [ ]:
# %%writefile app.py

# import streamlit as st
# import os
# import json
# import requests
# from langchain_community.document_loaders import PyMuPDFLoader
# from openai import OpenAI
# import tiktoken
# import pandas as pd
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_community.embeddings.openai import OpenAIEmbeddings
# from langchain_community.vectorstores import Chroma
# import tempfile


# OPENAI_API_KEY = os.environ.get("API_KEY")
# OPENAI_API_BASE = os.environ.get("API_BASE")

# # Initialize OpenAI client
# client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE
# )

# # Define the system prompt for the model
# qna_system_message = """
# You are an AI assistant designed to support research teams in efficiently reviewing scientific literature. Your task is to provide evidence-based, concise, and relevant summaries based on the context provided from research papers.

# User input will include the necessary context for you to answer their questions. This context will begin with the token:

# ###Context
# The context contains excerpts from one or more research papers, along with associated metadata such as titles, authors, abstracts, keywords, and specific sections relevant to the query.

# When crafting your response
# -Use only the provided context to answer the question.
# -If the answer is found in the context, respond with concise and insight-focused summaries.
# -Include the paper title and, where applicable, arXiv ID or section reference as the source.
# -If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


# Please adhere to the following response guidelines:
# -Provide clear, direct answers using only the given context.
# -Do not include any additional information outside of the context.
# -Avoid rephrasing or generalizing unless explicitly relevant to the question.
# -If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
# -If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."


# Here is an example of how to structure your response:

# Answer:
# [Answer based on context]

# Source:
# [Source details with page or section]
# """

# # Define the user message template
# qna_user_message_template = """
# ###Context
# Here are some excerpts from GEN AI Research Paper and their sources that are relevant to the Gen AI question mentioned below:
# {context}
# ###Question
# {question}
# """

# @st.cache_resource
# def load_and_process_pdfs(uploaded_files):
#     all_documents = []
#     for uploaded_file in uploaded_files:
#         with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
#             tmp_file.write(uploaded_file.getvalue())
#             tmp_file_path = tmp_file.name
#         loader = PyMuPDFLoader(tmp_file_path)
#         documents = loader.load()
#         all_documents.extend(documents)
#         os.remove(tmp_file_path) # Clean up the temporary file
#     text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#         encoding_name='cl100k_base',
#         chunk_size=1000,
#     )
#     document_chunks = text_splitter.split_documents(all_documents)

#     embedding_model = OpenAIEmbeddings(
#         openai_api_key=OPENAI_API_KEY,
#         openai_api_base=OPENAI_API_BASE
#     )

#     # Create an in-memory vector store (or use a persistent one if needed)
#     vectorstore = Chroma.from_documents(
#         document_chunks,
#         embedding_model
#     )
#     return vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5})

# def generate_rag_response(user_input, retriever, max_tokens=500, temperature=0, top_p=0.95):
#     # Retrieve relevant document chunks
#     relevant_document_chunks = retriever.get_relevant_documents(query=user_input)
#     context_list = [d.page_content for d in relevant_document_chunks]

#     # Combine document chunks into a single context
#     context_for_query = ". ".join(context_list)

#     user_message = qna_user_message_template.replace('{context}', context_for_query)
#     user_message = user_message.replace('{question}', user_input)

#     # Generate the response
#     try:
#         response = client.chat.completions.create(
#             model="gpt-4o-mini",
#             messages=[
#                 {"role": "system", "content": qna_system_message},
#                 {"role": "user", "content": user_message}
#             ],
#             max_tokens=max_tokens,
#             temperature=temperature,
#             top_p=top_p
#         )
#         response = response.choices[0].message.content.strip()
#     except Exception as e:
#         response = f'Sorry, I encountered the following error: \n {e}'

#     return response

# # Streamlit App
# st.title("LLM-Powered Research Assistant")

# uploaded_files = st.file_uploader("Upload PDF files", type=["pdf"], accept_multiple_files=True)

# retriever = None
# if uploaded_files:
#     st.info("Processing uploaded PDFs...")
#     retriever = load_and_process_pdfs(uploaded_files)
#     st.success("PDFs processed and ready for questioning!")


# if retriever:
#     user_question = st.text_input("Ask a question about the uploaded documents:")
#     if user_question:
#         with st.spinner("Generating response..."):
#             rag_response = generate_rag_response(user_question, retriever)
#             st.write(rag_response)

### Docker File

In [ ]:
# %%writefile Dockerfile

# FROM python:3.9-slim

# WORKDIR /app

# RUN apt-get update && apt-get install -y \
#     build-essential \
#     curl \
#     software-properties-common \
#     git \
#     && rm -rf /var/lib/apt/lists/*

# COPY requirements.txt ./
# COPY app.py ./
# COPY src/ ./src/

# RUN pip3 install -r requirements.txt

# EXPOSE 8501

# HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health

# ENTRYPOINT ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0","--server.enableXsrfProtection=false"]


### requirements.txt

In [ ]:
# %%writefile requirements.txt

# langchain_community==0.3.27
# langchain==0.3.27
# chromadb==1.0.15
# pymupdf==1.26.3
# tiktoken==0.9.0
# datasets==4.0.0
# evaluate==0.4.5
# streamlit==1.35.0
# openai==1.99.1
# langchain_openai==0.3.28

## Hugging Face Setup

#### 1. Login to Hugging Face

Go to [Hugging face](https://huggingface.co) and sign up or log in to your account.

#### 2. Create a New Space

   * Navigate to [Hugging face Spaces ](https://huggingface.co/spaces).
   * Click **Create New Space**.
   * Fill in:

     * Name for your Space.
     * Space SDK: Select **Docker**.
     * Choose a Docker template: Select **Streamlit**
     * Visibility: Choose *Public* or *Private*.
   * Click **Create Space**.

#### 3. Setup OpenAI tokens




- Open your Hugging Face Space

- Click on the **“Settings”** tab at the top of the Space.

- Scroll down to the **“Repository secrets”** section.

- Click **“Add a new secret”**.

- In the **Name** field, type token name

- In the **Secret** field, paste your secret key

- Click **“Add secret”** to save it.


#### 4. Upload Your Files




   * In the new Space, click the **Files** tab.
   * Delete existing requirements.txt , DockerFile
   * Click Contribute then Upload files and add:

     * `app.py`
     * `requirements.txt`
     * `DockerFile`
   * Commit the upload.

#### 5. Build and Launch



   * Hugging Face will automatically detect the `Dockerfile` and start building the container.
   * Wait a few minutes for the build to complete and the app to go live.

#### 6. Access and Test

* Go to the App tab or the Space URL to view and test your running Streamlit app.

